In [ ]:
import pandas as pd
import numpy as np
# extending dataset by adding more columns
data = pd.read_csv('insurance.csv')
n_samples = len(data)
# generating new features
chronic_conditions = np.random.poisson(lam=1.5, size=n_samples)  # count of chronic conditions (e.g., diabetes, hypertension)
alcohol_consumption = np.random.choice(['none', 'low', 'moderate', 'high'], size=n_samples, p=[0.2, 0.4, 0.3, 0.1])
exercise_frequency = np.random.choice(['none', 'rare', 'regular'], size=n_samples, p=[0.3, 0.4, 0.3])
employment_status = np.random.choice(['employed', 'unemployed', 'retired', 'student'], size=n_samples)
income_level = np.round(np.random.normal(50000, 15000, size=n_samples), -2)  # income in USD
insurance_type = np.random.choice(['private', 'medicaid', 'medicare', 'uninsured'], size=n_samples)
hospital_visits_last_year = np.random.poisson(lam=1.0, size=n_samples)
mental_health_score = np.round(np.random.normal(70, 15, size=n_samples))  # score from 0 to 100
education_level = np.random.choice(['high_school', 'bachelor', 'master', 'phd'], size=n_samples, p=[0.4, 0.3, 0.2, 0.1])
income_level = np.clip(income_level, 10000, 150000)
mental_health_score = np.clip(mental_health_score, 0, 100)
data_extended = data.copy()
data_extended['chronic_conditions'] = chronic_conditions
data_extended['alcohol_consumption'] = alcohol_consumption
data_extended['exercise_frequency'] = exercise_frequency
data_extended['employment_status'] = employment_status
data_extended['income_level'] = income_level
data_extended['insurance_type'] = insurance_type
data_extended['hospital_visits_last_year'] = hospital_visits_last_year
data_extended['mental_health_score'] = mental_health_score
data_extended['education_level'] = education_level
print(data_extended)
df = data_extended.copy()
file_path = 'healthcare_dataset.csv'
data_extended.to_csv(file_path, index=False)


      age     sex     bmi  children smoker     region      charges  \
0      19  female  27.900         0    yes  southwest  16884.92400   
1      18    male  33.770         1     no  southeast   1725.55230   
2      28    male  33.000         3     no  southeast   4449.46200   
3      33    male  22.705         0     no  northwest  21984.47061   
4      32    male  28.880         0     no  northwest   3866.85520   
...   ...     ...     ...       ...    ...        ...          ...   
1333   50    male  30.970         3     no  northwest  10600.54830   
1334   18  female  31.920         0     no  northeast   2205.98080   
1335   18  female  36.850         0     no  southeast   1629.83350   
1336   21  female  25.800         0     no  southwest   2007.94500   
1337   61  female  29.070         0    yes  northwest  29141.36030   

      chronic_conditions alcohol_consumption exercise_frequency  \
0                      0                high            regular   
1                      2 

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# encoding categorical variables using label encoder
categorical_columns = ['sex', 'smoker', 'region', 'alcohol_consumption', 'exercise_frequency', 
                       'employment_status', 'insurance_type', 'education_level']
label_encoders = {}
for column in categorical_columns:
    le = LabelEncoder()
    df[column] = le.fit_transform(df[column])
    label_encoders[column] = le

# numrical features
numerical_columns = ['age', 'bmi', 'children', 'income_level', 'hospital_visits_last_year', 'mental_health_score']
scaler = StandardScaler()
df[numerical_columns] = scaler.fit_transform(df[numerical_columns])


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib

def train_healthcare_model():
    df = pd.read_csv("healthcare_dataset.csv")
    categorical_columns = ['sex', 'smoker', 'region', 'alcohol_consumption',
                          'exercise_frequency', 'employment_status', 
                          'insurance_type', 'education_level']
    label_encoders = {}
    for col in categorical_columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        joblib.dump(le, f'label_encoder_{col}.pkl')
    numerical_columns = ['age', 'bmi', 'children', 'income_level',
                        'hospital_visits_last_year', 'mental_health_score']
    scaler = StandardScaler()
    df[numerical_columns] = scaler.fit_transform(df[numerical_columns])
    joblib.dump(scaler, 'scaler.pkl')
    X = df.drop(columns=['charges'])
    y = df['charges']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    joblib.dump(X_train.columns.tolist(), 'feature_names.pkl')
    joblib.dump(X_train, 'X_train.pkl')
    model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
    model.fit(X_train, y_train)
    joblib.dump(model, 'healthcare_cost_model.pkl')

if __name__ == "__main__":
    train_healthcare_model()
